# Soft Actor-Critic (SAC) — From Scratch in PyTorch

**Paper:** [Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor](https://arxiv.org/abs/1801.01290) (Haarnoja et al., 2018)

This notebook implements the SAC algorithm from scratch using PyTorch and trains it on the **Pendulum-v1** environment from Gymnasium.

## Key features implemented:
- **Twin Q-networks** (clipped double Q-learning to reduce overestimation bias)
- **Reparameterized tanh-Gaussian policy** (stochastic actor with differentiable sampling)
- **Automatic temperature (α) tuning** (target entropy = −|action_dim|)
- **Off-policy replay buffer** with uniform sampling
- **Polyak (soft) target updates**

## Simplifications:
- Single environment (Pendulum-v1) — no MuJoCo dependency
- Small MLPs (256 hidden units, 2 layers)
- 50,000 training steps (sufficient for Pendulum convergence)

In [1]:
# Install dependencies (Colab/Kaggle compatible)
import subprocess
subprocess.run(['pip', 'install', '-q', 'gymnasium', 'torch', 'numpy', 'matplotlib'], capture_output=True)

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque
import copy
import time

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.11.0+cu128
Using device: cuda


In [2]:
# === Hyperparameters ===
GAMMA = 0.99          # Discount factor
TAU = 0.005           # Polyak averaging coefficient for target networks
LR_CRITIC = 3e-4      # Learning rate for critic (Q-networks)
LR_ACTOR = 3e-4       # Learning rate for actor (policy)
LR_ALPHA = 3e-4       # Learning rate for temperature alpha
HIDDEN_DIM = 256      # Hidden layer width
BUFFER_SIZE = 1_000_000  # Replay buffer capacity
BATCH_SIZE = 256       # Mini-batch size for training
START_STEPS = 5_000    # Random exploration steps before training
TOTAL_STEPS = 50_000   # Total environment interaction steps
UPDATE_AFTER = 1_000   # Steps before training updates begin
UPDATE_EVERY = 50      # Steps between training rounds
NUM_UPDATES = 50       # Gradient steps per training round
EVAL_EVERY = 5_000     # Steps between evaluations
EVAL_EPISODES = 10     # Episodes per evaluation
SEED = 42

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print("Hyperparameters set.")

Hyperparameters set.


In [3]:
# === Replay Buffer ===
class ReplayBuffer:
    """Uniform-sampling replay buffer for off-policy RL."""
    def __init__(self, obs_dim, act_dim, size=1_000_000):
        self.obs_buf = np.zeros((size, obs_dim), dtype=np.float32)
        self.obs2_buf = np.zeros((size, obs_dim), dtype=np.float32)
        self.act_buf = np.zeros((size, act_dim), dtype=np.float32)
        self.rew_buf = np.zeros((size, 1), dtype=np.float32)
        self.done_buf = np.zeros((size, 1), dtype=np.float32)
        self.ptr = 0
        self.size = 0
        self.max_size = size

    def store(self, obs, act, rew, next_obs, done):
        self.obs_buf[self.ptr] = obs
        self.obs2_buf[self.ptr] = next_obs
        self.act_buf[self.ptr] = act
        self.rew_buf[self.ptr] = rew
        self.done_buf[self.ptr] = float(done)
        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)

    def sample(self, batch_size):
        idxs = np.random.randint(0, self.size, size=batch_size)
        return (
            torch.FloatTensor(self.obs_buf[idxs]).to(device),
            torch.FloatTensor(self.act_buf[idxs]).to(device),
            torch.FloatTensor(self.rew_buf[idxs]).to(device),
            torch.FloatTensor(self.obs2_buf[idxs]).to(device),
            torch.FloatTensor(self.done_buf[idxs]).to(device),
        )

print("ReplayBuffer defined.")

ReplayBuffer defined.


In [4]:
# === Actor: Tanh-Squashed Gaussian Policy ===
LOG_STD_MIN = -20
LOG_STD_MAX = 2

class GaussianPolicy(nn.Module):
    """
    Stochastic policy that outputs a tanh-squashed Gaussian action.
    Uses the reparameterization trick for differentiable sampling.
    """
    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim, act_dim)
        self.log_std_head = nn.Linear(hidden_dim, act_dim)

    def forward(self, obs):
        h = self.net(obs)
        mu = self.mu_head(h)
        log_std = self.log_std_head(h)
        log_std = torch.clamp(log_std, LOG_STD_MIN, LOG_STD_MAX)
        return mu, log_std

    def sample(self, obs):
        """
        Reparameterized sampling with log-probability correction for tanh.
        Returns: action (tanh-squashed), log_prob (entropy-corrected)
        """
        mu, log_std = self.forward(obs)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mu, std)
        # Reparameterization: z = mu + std * eps
        z = dist.rsample()
        # Squash to [-1, 1]
        action = torch.tanh(z)
        # Log probability with tanh correction:
        # log_prob = log N(z | mu, std) - sum(log(1 - tanh(z)^2))
        log_prob = dist.log_prob(z)
        # Enforce action bounds for numerical stability
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        return action, log_prob

    def deterministic_action(self, obs):
        mu, _ = self.forward(obs)
        return torch.tanh(mu)

print("GaussianPolicy (Actor) defined.")

GaussianPolicy (Actor) defined.


In [5]:
# === Critic: Twin Q-Networks ===
class QNetwork(nn.Module):
    """Single Q-network: maps (state, action) -> Q-value."""
    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim + act_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.net(x)


class TwinQ(nn.Module):
    """
    Holds two independent Q-networks (Q1, Q2) and their target copies.
    Uses clipped double Q-learning: takes min(Q1, Q2) for target values.
    """
    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.Q1 = QNetwork(obs_dim, act_dim, hidden_dim)
        self.Q2 = QNetwork(obs_dim, act_dim, hidden_dim)
        self.Q1_target = QNetwork(obs_dim, act_dim, hidden_dim)
        self.Q2_target = QNetwork(obs_dim, act_dim, hidden_dim)
        # Initialize targets to match main networks
        self.Q1_target.load_state_dict(self.Q1.state_dict())
        self.Q2_target.load_state_dict(self.Q2.state_dict())
        # Freeze target gradients
        for p in self.Q1_target.parameters():
            p.requires_grad = False
        for p in self.Q2_target.parameters():
            p.requires_grad = False

    def forward(self, obs, act):
        return self.Q1(obs, act), self.Q2(obs, act)

    def Q1_val(self, obs, act):
        return self.Q1(obs, act)

    def min_target(self, obs, act):
        return torch.min(self.Q1_target(obs, act), self.Q2_target(obs, act))

    def update_targets(self, tau):
        """Polyak averaging: target <- (1-tau)*target + tau*main"""
        with torch.no_grad():
            for p_t, p in zip(self.Q1_target.parameters(), self.Q1.parameters()):
                p_t.data.mul_(1 - tau).add_(tau * p.data)
            for p_t, p in zip(self.Q2_target.parameters(), self.Q2.parameters()):
                p_t.data.mul_(1 - tau).add_(tau * p.data)

print("TwinQ (Critic) defined.")

TwinQ (Critic) defined.


In [6]:
# === SAC Agent ===
class SACAgent:
    """
    Soft Actor-Critic agent with:
    - Twin Q-networks (clipped double Q-learning)
    - Reparameterized tanh-Gaussian policy
    - Automatic temperature (alpha) tuning
    """
    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        self.obs_dim = obs_dim
        self.act_dim = act_dim

        # Actor (policy)
        self.actor = GaussianPolicy(obs_dim, act_dim, hidden_dim).to(device)
        self.actor_optim = Adam(self.actor.parameters(), lr=LR_ACTOR)

        # Critic (twin Q)
        self.critic = TwinQ(obs_dim, act_dim, hidden_dim).to(device)
        self.critic_optim = Adam(
            list(self.critic.Q1.parameters()) + list(self.critic.Q2.parameters()),
            lr=LR_CRITIC
        )

        # Automatic temperature tuning
        self.target_entropy = -float(act_dim)
        self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
        self.log_alpha_optim = Adam([self.log_alpha], lr=LR_ALPHA)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def act(self, obs, deterministic=False):
        """Select action for environment interaction."""
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            if deterministic:
                action = self.actor.deterministic_action(obs_t)
            else:
                action, _ = self.actor.sample(obs_t)
        return action.cpu().numpy()[0]

    def update(self, data):
        """One gradient step on critic, actor, and alpha."""
        obs, act, rew, next_obs, done = data

        # ---- 1. Critic Update ----
        with torch.no_grad():
            next_act, next_log_prob = self.actor.sample(next_obs)
            next_q = self.critic.min_target(next_obs, next_act)
            next_v = next_q - self.alpha * next_log_prob
            target_q = rew + GAMMA * (1 - done) * next_v

        q1, q2 = self.critic(obs, act)
        critic_loss = F.mse_loss(q1, target_q) + F.mse_loss(q2, target_q)

        self.critic_optim.zero_grad()
        critic_loss.backward()
        self.critic_optim.step()

        # ---- 2. Actor Update ----
        new_act, log_prob = self.actor.sample(obs)
        q1_new = self.critic.Q1_val(obs, new_act)
        q2_new = self.critic.Q2(obs, new_act)
        min_q = torch.min(q1_new, q2_new)
        actor_loss = (self.alpha * log_prob - min_q).mean()

        self.actor_optim.zero_grad()
        actor_loss.backward()
        self.actor_optim.step()

        # ---- 3. Alpha (temperature) Update ----
        alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()

        self.log_alpha_optim.zero_grad()
        alpha_loss.backward()
        self.log_alpha_optim.step()

        # ---- 4. Target Network Soft Update ----
        self.critic.update_targets(TAU)

        return {
            'critic_loss': critic_loss.item(),
            'actor_loss': actor_loss.item(),
            'alpha_loss': alpha_loss.item(),
            'alpha': self.alpha.item(),
            'entropy': -log_prob.mean().item(),
        }

print("SACAgent defined.")

SACAgent defined.


In [7]:
# === Environment Setup ===
env = gym.make('Pendulum-v1')
eval_env = gym.make('Pendulum-v1')

obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
act_low = env.action_space.low
act_high = env.action_space.high

print(f"Environment: Pendulum-v1")
print(f"Observation dim: {obs_dim}")
print(f"Action dim: {act_dim}")
print(f"Action range: [{act_low}, {act_high}]")

agent = SACAgent(obs_dim, act_dim, hidden_dim=HIDDEN_DIM)
replay_buffer = ReplayBuffer(obs_dim, act_dim, size=BUFFER_SIZE)

print(f"Agent initialized. Target entropy: {agent.target_entropy}")

Environment: Pendulum-v1
Observation dim: 3
Action dim: 1
Action range: [[-2.], [2.]]


Agent initialized. Target entropy: -1.0


In [8]:
# === Training Loop ===
start_time = time.time()

# Logging containers
train_rewards = []
train_episodes = []
eval_rewards = []
eval_steps = []
critic_losses = []
actor_losses = []
alpha_values = []
entropy_values = []
log_steps = []

obs, _ = env.reset(seed=SEED)
episode_reward = 0
episode_len = 0
episode_count = 0

for step in range(1, TOTAL_STEPS + 1):
    # --- Collect experience ---
    if step <= START_STEPS:
        action = env.action_space.sample()
    else:
        action = agent.act(obs, deterministic=False)

    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    replay_buffer.store(obs, action, reward, next_obs, done)

    obs = next_obs
    episode_reward += reward
    episode_len += 1

    if done:
        train_rewards.append(episode_reward)
        train_episodes.append(episode_count)
        episode_count += 1
        episode_reward = 0
        episode_len = 0
        obs, _ = env.reset()

    # --- Training updates ---
    if step >= UPDATE_AFTER and step % UPDATE_EVERY == 0:
        for _ in range(NUM_UPDATES):
            data = replay_buffer.sample(BATCH_SIZE)
            metrics = agent.update(data)
            if step % 500 == 0:
                critic_losses.append(metrics['critic_loss'])
                actor_losses.append(metrics['actor_loss'])
                alpha_values.append(metrics['alpha'])
                entropy_values.append(metrics['entropy'])
                log_steps.append(step)

    # --- Evaluation ---
    if step % EVAL_EVERY == 0:
        eval_reward = 0
        for _ in range(EVAL_EPISODES):
            eval_obs, _ = eval_env.reset()
            eval_done = False
            while not eval_done:
                eval_action = agent.act(eval_obs, deterministic=True)
                eval_obs, eval_r, eval_term, eval_trunc, _ = eval_env.step(eval_action)
                eval_reward += eval_r
                eval_done = eval_term or eval_trunc
        avg_eval_reward = eval_reward / EVAL_EPISODES
        eval_rewards.append(avg_eval_reward)
        eval_steps.append(step)
        elapsed = time.time() - start_time
        print(f"Step {step:>6d} | Eval Reward: {avg_eval_reward:>8.2f} | "
              f"Alpha: {metrics['alpha']:.4f} | Entropy: {metrics['entropy']:.3f} | "
              f"Time: {elapsed:.1f}s")

print(f"\nTraining complete! Total time: {time.time() - start_time:.1f}s")
print(f"Episodes completed: {episode_count}")

Step   5000 | Eval Reward:  -403.18 | Alpha: 0.3463 | Entropy: -0.295 | Time: 39.2s


Step  10000 | Eval Reward:  -267.72 | Alpha: 0.1535 | Entropy: -1.133 | Time: 89.3s


Step  15000 | Eval Reward:  -397.98 | Alpha: 0.1418 | Entropy: -0.869 | Time: 139.3s


Step  20000 | Eval Reward:  -479.16 | Alpha: 0.1378 | Entropy: -0.968 | Time: 189.4s


Step  25000 | Eval Reward:  -292.68 | Alpha: 0.1562 | Entropy: -1.088 | Time: 239.9s


Step  30000 | Eval Reward:  -262.61 | Alpha: 0.1142 | Entropy: -0.952 | Time: 290.1s


Step  35000 | Eval Reward:  -170.07 | Alpha: 0.0966 | Entropy: -0.874 | Time: 340.2s


Step  40000 | Eval Reward:  -216.80 | Alpha: 0.0841 | Entropy: -1.073 | Time: 390.3s


Step  45000 | Eval Reward:  -229.40 | Alpha: 0.0750 | Entropy: -1.161 | Time: 440.3s


Step  50000 | Eval Reward:  -210.43 | Alpha: 0.0643 | Entropy: -0.606 | Time: 490.0s

Training complete! Total time: 490.0s
Episodes completed: 250


In [9]:
# === Visualization ===

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Soft Actor-Critic (SAC) on Pendulum-v1', fontsize=16, fontweight='bold')

# Plot 1: Training Reward Curve
ax1 = axes[0, 0]
ax1.plot(train_episodes, train_rewards, alpha=0.3, color='blue', label='Episode reward')
if len(train_rewards) > 10:
    window = min(20, len(train_rewards) // 3)
    rolling = np.convolve(train_rewards, np.ones(window)/window, mode='valid')
    ax1.plot(train_episodes[window-1:], rolling, color='red', linewidth=2, label=f'Rolling avg ({window})')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward')
ax1.set_title('Training Reward Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Plot 2: Evaluation Reward Curve
ax2 = axes[0, 1]
ax2.plot(eval_steps, eval_rewards, 'g-o', linewidth=2, markersize=4)
ax2.set_xlabel('Environment Steps')
ax2.set_ylabel('Avg Evaluation Reward')
ax2.set_title('Evaluation Reward Curve')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Plot 3: Alpha (temperature) over training
ax3 = axes[1, 0]
ax3.plot(log_steps, alpha_values, 'orange', linewidth=2)
ax3.set_xlabel('Environment Steps')
ax3.set_ylabel('Alpha (temperature)')
ax3.set_title('Automatic Temperature (α) Tuning')
ax3.grid(True, alpha=0.3)

# Plot 4: Policy Entropy over training
ax4 = axes[1, 1]
ax4.plot(log_steps, entropy_values, 'purple', linewidth=2, label='Policy entropy')
ax4.axhline(y=agent.target_entropy, color='red', linestyle='--', linewidth=1.5, label=f'Target entropy ({agent.target_entropy:.0f})')
ax4.set_xlabel('Environment Steps')
ax4.set_ylabel('Entropy')
ax4.set_title('Policy Entropy vs Target')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sac_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training plots saved.")

Training plots saved.


In [10]:
# === Critic and Actor Loss Curves ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(log_steps, critic_losses, 'b-', alpha=0.7)
ax1.set_xlabel('Environment Steps')
ax1.set_ylabel('Critic Loss (MSE)')
ax1.set_title('Critic (Twin Q) Loss')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(log_steps, actor_losses, 'r-', alpha=0.7)
ax2.set_xlabel('Environment Steps')
ax2.set_ylabel('Actor Loss')
ax2.set_title('Actor (Policy) Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sac_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Loss curves saved.")

Loss curves saved.


In [11]:
# === Final Evaluation ===
final_eval_rewards = []
for ep in range(20):
    eval_obs, _ = eval_env.reset()
    eval_done = False
    ep_reward = 0
    while not eval_done:
        eval_action = agent.act(eval_obs, deterministic=True)
        eval_obs, eval_r, eval_term, eval_trunc, _ = eval_env.step(eval_action)
        ep_reward += eval_r
        eval_done = eval_term or eval_trunc
    final_eval_rewards.append(ep_reward)

print(f"=== Final Evaluation (20 episodes, deterministic policy) ===")
print(f"Mean reward:   {np.mean(final_eval_rewards):.2f}")
print(f"Std reward:    {np.std(final_eval_rewards):.2f}")
print(f"Min reward:    {np.min(final_eval_rewards):.2f}")
print(f"Max reward:    {np.max(final_eval_rewards):.2f}")
print(f"\nFinal alpha: {agent.alpha.item():.4f}")
print(f"Target entropy: {agent.target_entropy}")

# Pendulum-v1 reward range: approximately [-1600, 0]
# A good SAC agent should achieve average reward > -250 (ideally > -200)
if np.mean(final_eval_rewards) > -300:
    print("\n✅ SAC successfully learned to solve Pendulum-v1!")
else:
    print("\n⚠️ SAC performance is below expected. May need more training steps.")

=== Final Evaluation (20 episodes, deterministic policy) ===
Mean reward:   -236.32
Std reward:    159.22
Min reward:    -490.04
Max reward:    -0.13

Final alpha: 0.0643
Target entropy: -1.0

✅ SAC successfully learned to solve Pendulum-v1!


## Summary

This notebook implements the **Soft Actor-Critic (SAC)** algorithm from scratch in PyTorch:

| Component | Implementation |
|-----------|---------------|
| **Actor** | Tanh-squashed Gaussian policy with reparameterization trick |
| **Critic** | Twin Q-networks with clipped double Q-learning |
| **Temperature α** | Automatic tuning via dual gradient descent |
| **Target updates** | Polyak (soft) averaging with τ=0.005 |
| **Replay buffer** | Uniform sampling, 1M capacity |
| **Environment** | Pendulum-v1 (Gymnasium) |

### Key results:
- The agent should achieve an average evaluation reward better than −300 on Pendulum-v1.
- The policy entropy should track near the target entropy (−1 for 1D action space).
- The temperature α should auto-adjust during training without manual tuning.

### Algorithm pseudocode (simplified):
```
for each training step:
    1. Sample action from π(a|s), execute in env, store (s,a,r,s',done) in buffer
    2. Sample batch from replay buffer
    3. Critic update:
       a', log_π' = actor.sample(s')
       V_target = min(Q1_tg(s',a'), Q2_tg(s',a')) - α·log_π'
       y = r + γ·(1-done)·V_target
       L_critic = MSE(Q1(s,a), y) + MSE(Q2(s,a), y)
    4. Actor update:
       a_new, log_π = actor.sample(s)
       L_actor = (α·log_π - min(Q1(s,a_new), Q2(s,a_new))).mean()
    5. Alpha update:
       L_α = -(log_α · (log_π + target_entropy).detach()).mean()
    6. Soft update targets: θ_tg ← (1-τ)·θ_tg + τ·θ
```

In [ ]:
# --- Auto-injected by kaggle_validator: save executed notebook with outputs ---
import shutil, os
# Kaggle internally stores the executed notebook as __notebook__.ipynb
if os.path.exists('/kaggle/working/__notebook__.ipynb'):
    shutil.copy('/kaggle/working/__notebook__.ipynb', '/kaggle/working/executed_solution.ipynb')
    print('Executed notebook saved to output folder.')
else:
    # Fallback: try alternative path
    import glob
    candidates = glob.glob('/kaggle/working/*.ipynb')
    print(f'Available notebooks in output: {candidates}')
